# Gợi ý lọc dựa trên nội dung - Độ tương đồng nút (Node Similarity)

## Introduction


Tính toán độ tương đồng giữa các cặp nút Poi, hoàn toàn dựa trên nội dung (các thuộc tính) của chính Poi đó, không liên quan đến tương tác của người dùng.

Các độ đo tương đồng:
- Thuộc tính số: Euclidean Distance Similarity
- Thuộc tính phân loại: Jaccard Similarity
- Thuộc tính văn bản: Cosine Similarity

Độ tương đồng tổng thể được tính bằng cách kết hợp tất cả các thuộc tính trên cơ sở trọng số.

Cắt bỏ điểm tương đồng dưới 0.5 để đảm bảo tất cả các cặp đều tương đồng.

Tạo gợi ý bằng cách tìm các POI tương đồng nhất.

## Điều kiện tiên quyết

Neo4j server đã được cài đặt phiên bản GDS mới (2.0+).

Thư viện Python `graphdatascience` để vận hành Neo4j GDS.

Truy vấn `Cypher` để tạo các gợi ý.

Gói `py2neo` để ghi ngược pandas dataframe vào cơ sở dữ liệu neo4j.

In [1]:
import os
import configparser

import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import jaccard_score

from py2neo import Graph
from graphdatascience import GraphDataScience

import requests
import time

import torch
from transformers import AutoModel, AutoTokenizer
from pyvi import ViTokenizer
import numpy as np

d:\Thac_Si\De_an_thac_si\code\KG-Rec-Sys-Tourism-SG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Sử dụng file ini cho thông tin đăng nhập, nếu không sử dụng mặc định
HOST = 'neo4j://localhost'
USERNAME = 'neo4j'
DATABASE = 'neo4j'
PASSWORD = 'password'

NEO4J_CONF_FILE = 'neo4j.ini'

if NEO4J_CONF_FILE is not None and os.path.exists(NEO4J_CONF_FILE):
    config = configparser.RawConfigParser()
    config.read(NEO4J_CONF_FILE)
    HOST = config['NEO4J']['HOST']
    DATABASE = config['NEO4J'].get('DATABASE', 'neo4j')
    USERNAME = config['NEO4J'].get('USERNAME', DATABASE)
    PASSWORD = config['NEO4J']['PASSWORD']
    print(f'Using custom database properties \nHOST: {HOST}; DATABASE: {DATABASE}; PASSWORD: {PASSWORD}')
else:
    print('Could not find database properties file, using defaults')

# Kết nối với cơ sở dữ liệu Neo4j bằng thư viện GDS
gds = GraphDataScience(HOST, auth=(USERNAME, PASSWORD))
gds.set_database(DATABASE)

# Kết nối tới cơ sở dữ liệu Neo4j bằng py2neo
graph = Graph(HOST, auth=(USERNAME, PASSWORD), name=DATABASE)

Using custom database properties 
HOST: neo4j://127.0.0.1:7687; DATABASE: neo4j; PASSWORD: 12345678


# Tính toán độ tương đồng nút giữa các cặp POI

In [3]:
# Trích xuất dữ liệu thô của nút POI và các thuộc tính của nó từ GDS
result = gds.run_cypher("""
MATCH (poi:Poi)
OPTIONAL MATCH (poi)-[:BELONGS_TO]->(category:Category)
OPTIONAL MATCH (poi)-[:LOCATED_AT]->(region:Region)
RETURN poi.id AS poi_id, 
       poi.name AS name, 
                        
       poi.description AS description, 

       poi.openingHours AS opening_hours, 
       poi.duration AS duration, 
       category.name AS category, 
       region.name AS region,
                        
       poi.price AS price, 
       poi.avgRating AS avg_rating, 
       poi.numReviews AS num_reviews, 
       poi.numReviews_5 AS num_reviews_5, 
       poi.numReviews_4 AS num_reviews_4, 
       poi.numReviews_3 AS num_reviews_3, 
       poi.numReviews_2 AS num_reviews_2, 
       poi.numReviews_1 AS num_reviews_1
""")

# Chuyển đổi kết quả thành DataFrame
df_pois = pd.DataFrame(result)

df_pois

,poi_id,name,description,opening_hours,duration,category,region,price,avg_rating,num_reviews,num_reviews_5,num_reviews_4,num_reviews_3,num_reviews_2,num_reviews_1
0,311103,War Remnants Museum,Bảo tàng Chứng tích Chiến tranh là đơn vị trực...,Mo-Su 07:30-17:30,1-2 giờ,Viện bảo tàng lịch sử,Thành phố Hồ Chí Minh,0.0,4.4,33656,17384,11986,3435,566,285
1,2005826,Cu Chi Tunnels,Trong chuyến tham quan có hướng dẫn đến Địa đạ...,Mo-Su 07:00-17:00,2-3 giờ,Điểm thu hút khách tham quan & thắng cảnh,Thành phố Hồ Chí Minh,0.0,4.4,26270,14611,8604,2309,479,267
2,311087,Chợ Bến Thành,Chợ trung tâm này là biểu tượng và linh hồn củ...,Mo-Su 09:00-01:00,1-2 giờ,Chợ trời & chợ đường phố,Quận 1,0.0,3.4,13499,2476,4339,4240,1492,952
3,4542125,À Ố Show - Lune Production,À Ố Show là một trong những chương trình biểu ...,Mo-Su 09:30-18:00,1-2 giờ,Rạp hát,Thành phố Hồ Chí Minh,0.0,4.6,3042,2278,549,150,40,25
4,311089,Bưu điện trung tâm Sài Gòn,Bưu điện này cũng có một trung tâm văn phòng h...,"['Mo-Sa 07:30-18:00', 'Su 08:00-17:00']",Dưới 1 giờ,Tòa nhà kiến trúc • Tòa nhà chính phủ,Thành phố Hồ Chí Minh,0.0,4.0,17418,5289,7538,3947,527,117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3400,34509929,Phiêu - KiN Rooftop Pool Bar,,Mo-Su 07:00-00:00,,Quán bar & câu lạc bộ,Thành phố Hồ Chí Minh,0.0,0.0,0,0,0,0,0,0
3401,34508995,EMO SPA - Quận 3,,Mo-Su 10:00-21:30,,Spa,Thành phố Hồ Chí Minh,0.0,0.0,1,1,0,0,0,0
3402,34515006,Siêu thị WinMart Landmark 81,,Mo-Su 08:30-22:00,,Cửa hàng đặc sản & cửa hàng quà tặng,Thành phố Hồ Chí Minh,0.0,0.0,0,0,0,0,0,0
3403,34435391,Sol Cu Chi Cooking Class,,Mo-Su 07:00-22:00,,Chuyến tham quan văn hóa • Lớp nấu ăn,Thành phố Hồ Chí Minh,0.0,0.0,0,0,0,0,0,0


In [4]:
# Trích xuất poi_id và poi_name duy nhất
df_distinct_pois = df_pois.copy()
df_distinct_pois = df_distinct_pois[['poi_id', 'name']].drop_duplicates()

df_distinct_pois

,poi_id,name
0,311103,War Remnants Museum
1,2005826,Cu Chi Tunnels
2,311087,Chợ Bến Thành
3,4542125,À Ố Show - Lune Production
4,311089,Bưu điện trung tâm Sài Gòn
...,...,...
3400,34509929,Phiêu - KiN Rooftop Pool Bar
3401,34508995,EMO SPA - Quận 3
3402,34515006,Siêu thị WinMart Landmark 81
3403,34435391,Sol Cu Chi Cooking Class


In [5]:
# Thuộc tính số - Chuẩn hóa Min-Max
# Các thuộc tính: 'avg_rating', 'num_reviews', 'num_reviews_5', 'num_reviews_4', 'num_reviews_3', 'num_reviews_2', 'num_reviews_1'

scaler = MinMaxScaler()
numerical_cols = ['avg_rating', 'num_reviews', 'num_reviews_5', 'num_reviews_4', 'num_reviews_3', 'num_reviews_2', 'num_reviews_1']
# Tạo một DataFrame mới với các cột đã chuẩn hóa và poi_id
df_numerical_cols = df_pois.copy()
df_numerical_cols = df_numerical_cols[['poi_id', 'name'] + numerical_cols]
# chỉ giữ lại các bản ghi duy nhất
df_numerical_cols = df_numerical_cols.drop_duplicates()
# Điền các giá trị thiếu trong cột số bằng 0
df_numerical_cols = df_numerical_cols.fillna(0)

# chuẩn hóa thuộc tính
df_numerical_cols[numerical_cols] = scaler.fit_transform(df_numerical_cols[numerical_cols])

df_numerical_cols

,poi_id,name,avg_rating,num_reviews,num_reviews_5,num_reviews_4,num_reviews_3,num_reviews_2,num_reviews_1
0,311103,War Remnants Museum,0.88,0.637775,0.357688,1.000000,0.810142,0.379357,0.299370
1,2005826,Cu Chi Tunnels,0.88,0.497811,0.300632,0.717837,0.544575,0.321046,0.280462
2,311087,Chợ Bến Thành,0.68,0.255803,0.050945,0.362006,1.000000,1.000000,1.000000
3,4542125,À Ố Show - Lune Production,0.92,0.057645,0.046871,0.045803,0.035377,0.026810,0.026261
4,311089,Bưu điện trung tâm Sài Gòn,0.80,0.330068,0.108825,0.628900,0.930896,0.353217,0.122899
...,...,...,...,...,...,...,...,...,...
3400,34509929,Phiêu - KiN Rooftop Pool Bar,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3401,34508995,EMO SPA - Quận 3,0.00,0.000019,0.000021,0.000000,0.000000,0.000000,0.000000
3402,34515006,Siêu thị WinMart Landmark 81,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3403,34435391,Sol Cu Chi Cooking Class,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [6]:
# Thuộc tính phân loại - Mã hóa One-hot
# Các thuộc tính: category, region, opening_Hours, duration

categorical_cols = ['category', 'region', 'opening_hours', 'duration']

# Sao chép df_pois chỉ với các cột phân loại được chỉ định
df_categorical_cols = df_pois.copy()
df_categorical_cols = df_categorical_cols[['poi_id', 'name'] + categorical_cols]

# Thực hiện mã hóa one-hot cho các cột phân loại
df_categorical_cols = pd.get_dummies(df_categorical_cols, columns=categorical_cols)


# Gộp các dòng có cùng poi_id bằng phép toán logic OR
df_categorical_cols = df_categorical_cols.groupby('poi_id').max().reset_index()

df_categorical_cols

,poi_id,name,category_ATV & xe địa hình,category_ATV & xe địa hình • Chuyến tham quan bằng mô tô,category_ATV & xe địa hình • Chuyến tham quan bằng xe buýt du lịch,category_ATV & xe địa hình • Chuyến tham quan bằng xe đạp,category_ATV & xe địa hình • Taxi & xe đưa đón,category_Bãi biển,category_Buổi học & hội thảo,category_Buổi học & hội thảo • Cho thuê tiện ích,...,"opening_hours_['Tu-Th 19:00-01:00', 'Fr-Su 19:00-02:00']","opening_hours_['We,Su 09:00-17:00', 'Sa 00:00-23:59', 'Su 18:00-21:00']","opening_hours_['We-Fr 15:00-21:00', 'Sa-Su 11:00-21:00']","opening_hours_['We-Sa 09:00-16:30', 'Su 09:00-11:30']","opening_hours_['We-Su 16:00-00:00', 'Fr 12:00-03:00', 'Sa-Su 12:00-06:00']",duration_,duration_1-2 giờ,duration_2-3 giờ,duration_Dưới 1 giờ,duration_Hơn 3 giờ
0,311087,Chợ Bến Thành,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
1,311089,Bưu điện trung tâm Sài Gòn,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,311092,Cong Vien Van Hoa Park,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
3,311094,Chùa Ngọc Hoàng (Phước Hải Tự),False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
4,311100,Bảo Tàng Thành Phố Hồ Chí Minh,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3321,34515006,Siêu thị WinMart Landmark 81,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
3322,34516489,LamViet Tattoo Shop - Hochiminh Tattoo,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
3323,34518573,Roots - Organic Store & Juice Bar,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
3324,34524442,Hotpot Comedy,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False


In [7]:
# # Thuộc tính văn bản - Đếm Token
# # Các thuộc tính: description

# textual_cols = ['description']

# # Sao chép df_pois chỉ với các cột văn bản được chỉ định
# df_cols = df_pois.copy()
# df_cols = df_cols[['poi_id', 'name'] + textual_cols]

# # chỉ giữ lại các bản ghi duy nhất
# df_cols = df_cols.drop_duplicates()

# # tạo mặt nạ để kiểm tra xem cột description có chứa chuỗi rỗng không
# df_cols['description'] = df_cols['description'].fillna('NULL')
# empty_description = df_cols['description'] == ''

# # Điền các chuỗi rỗng bằng "NULL"
# df_cols.loc[empty_description, 'description'] = 'NULL'


# # Tải danh sách stop word tiếng Việt chuẩn từ Github
# url = "https://raw.githubusercontent.com/stopwords/vietnamese-stopwords/master/vietnamese-stopwords.txt"
# response = requests.get(url)
# vietnamese_stopwords = response.text.splitlines()

# # Khởi tạo CountVectorizer tạm thời để lấy bộ phân tách từ (analyzer)
# temp_vectorizer = CountVectorizer()
# analyzer = temp_vectorizer.build_analyzer()

# # Phân tách danh sách từ dừng gốc thành các từ đơn nhất quán
# clean_stopwords = set()
# for word in vietnamese_stopwords:
#     # Tách từ ghép thành các từ đơn lẻ giống cách CountVectorizer sẽ làm với văn bản
#     clean_stopwords.update(analyzer(word))

# # khởi tạo bộ đếm token, bỏ qua stop words
# count_vectorizer = CountVectorizer(stop_words=list(clean_stopwords))

# # Tạo một DataFrame trống để lưu trữ số lượng token
# df_textual_cols = pd.DataFrame()

# # Lặp qua từng POI và mô tả của nó
# for index, row in df_cols.iterrows():

#     # Phân tách từ (tokenize) mô tả
#     description = [row['description']]
#     # print(f'description: {description}')
    
#     # Đếm token và lưu trữ trong ma trận thưa, sau đó chuyển sang ma trận dày
#     sparse_matrix = count_vectorizer.fit_transform(description)
#     doc_term_matrix = sparse_matrix.todense()
    
#     # Tạo DataFrame từ ma trận dày
#     df_token_counts = pd.DataFrame(
#         doc_term_matrix,
#         columns=count_vectorizer.get_feature_names_out(),
#         index=[row['poi_id']]
#     )
    
#     # Nối DataFrame vào df_token_counts
#     df_textual_cols = pd.concat([df_textual_cols, df_token_counts])

# # Khởi tạo lại chỉ mục, đổi tên thành poi_id, và điền các giá trị NaN bằng 0
# df_textual_cols.reset_index(inplace=True)
# df_textual_cols = df_textual_cols.rename(columns={'index': 'poi_id'})
# df_textual_cols = df_textual_cols.fillna(0)

# df_textual_cols

In [7]:
# Thuộc tính văn bản - PhoBERT Embedding
# Các thuộc tính: description

textual_cols = ['description']

# Sao chép df_pois chỉ với các cột văn bản được chỉ định
df_cols = df_pois.copy()
df_cols = df_cols[['poi_id', 'name'] + textual_cols]

# chỉ giữ lại các bản ghi duy nhất
df_cols = df_cols.drop_duplicates()

# điền các giá trị thiếu hoặc rỗng bằng "NULL"
df_cols['description'] = df_cols['description'].fillna('NULL')
empty_description = df_cols['description'].str.strip() == ''
df_cols.loc[empty_description, 'description'] = 'NULL'

# Tải mô tả
descriptions = df_cols['description'].tolist()

# Tokenize tiếng Việt bằng PyVi
print("Đang tiến hành tách từ (word segmentation)...")
segmented_descriptions = [ViTokenizer.tokenize(desc) for desc in descriptions]

# Cấu hình thiết bị GPU/CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {device}")

# Khởi tạo Tokenizer và PhoBERT model
print("Đang tải mô hình PhoBERT...")
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
phobert = AutoModel.from_pretrained("vinai/phobert-base-v2").to(device)
phobert.eval()

# Hàm trích xuất embedding
def get_phobert_embeddings(texts, batch_size=32):
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]
            # Tokenize và padding
            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            ).to(device)
            
            # Feed forward
            outputs = phobert(**inputs)
            
            # Sử dụng vector biểu diễn của token [CLS] (ở index 0) làm sentence embedding
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)
            
            if (i + batch_size) % (batch_size * 10) == 0 or (i + batch_size) >= len(texts):
                print(f"Đã xử lý {min(i + batch_size, len(texts))}/{len(texts)} mô tả...")
                
    return np.vstack(embeddings)

print("Đang trích xuất embedding...")
embeddings = get_phobert_embeddings(segmented_descriptions)

# Tạo DataFrame lưu trữ vector embedding (768 chiều)
df_textual_cols = pd.DataFrame(embeddings)
# Đặt lại cột poi_id làm cột đầu tiên
df_textual_cols.insert(0, 'poi_id', df_cols['poi_id'].values)

print(f"Hoàn thành! Kích thước ma trận embedding: {df_textual_cols.shape}")
df_textual_cols

Đang tiến hành tách từ (word segmentation)...
Sử dụng thiết bị: cuda
Đang tải mô hình PhoBERT...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 63175.92it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Đang trích xuất embedding...
Đã xử lý 320/3326 mô tả...
Đã xử lý 640/3326 mô tả...
Đã xử lý 960/3326 mô tả...
Đã xử lý 1280/3326 mô tả...
Đã xử lý 1600/3326 mô tả...
Đã xử lý 1920/3326 mô tả...
Đã xử lý 2240/3326 mô tả...
Đã xử lý 2560/3326 mô tả...
Đã xử lý 2880/3326 mô tả...
Đã xử lý 3200/3326 mô tả...
Đã xử lý 3326/3326 mô tả...
Hoàn thành! Kích thước ma trận embedding: (3326, 769)


,poi_id,0,1,2,3,4,5,6,7,8,...,758,759,760,761,762,763,764,765,766,767
0,311103,0.277710,0.780890,-0.134664,-0.410917,-0.106398,0.029093,-0.340525,0.267970,-0.036326,...,-0.353709,-0.097179,-0.204252,0.317571,0.006270,0.080136,0.074033,-0.094150,0.314926,0.264632
1,2005826,0.223623,0.461423,-0.229553,0.001595,0.080330,-0.100201,-0.373674,-0.056609,-0.184677,...,-0.078149,-0.106070,-0.349972,0.197380,-0.115617,-0.076882,-0.151177,0.010723,0.355564,0.112017
2,311087,0.350275,0.448508,-0.071413,-0.353737,0.052011,-0.054378,-0.237745,0.137912,-0.225874,...,-0.180798,-0.329003,-0.519825,0.443172,0.033020,-0.129983,-0.263809,-0.375161,0.258627,0.204178
3,4542125,0.147545,0.516198,-0.074098,-0.499945,-0.183398,0.048189,-0.511132,-0.168152,-0.145218,...,-0.304658,-0.309903,0.156395,0.372285,-0.059904,0.172600,0.075971,-0.087815,0.449511,0.120198
4,311089,0.227081,0.309607,-0.495314,-0.433226,-0.111259,-0.169163,0.059750,0.121620,-0.211863,...,-0.226206,-0.411791,0.089009,0.320216,-0.124864,0.049679,0.009453,0.081602,0.172914,0.188853
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3321,34509929,0.103390,0.656023,-0.262988,-0.743587,-0.146896,-0.067323,-0.100425,0.071572,-0.222507,...,0.117930,-0.324336,0.203228,0.444411,0.160399,-0.122193,-0.094062,-0.154989,0.303665,0.201930
3322,34508995,0.103390,0.656023,-0.262988,-0.743587,-0.146896,-0.067323,-0.100425,0.071572,-0.222507,...,0.117930,-0.324336,0.203228,0.444411,0.160399,-0.122193,-0.094062,-0.154989,0.303665,0.201930
3323,34515006,0.103390,0.656023,-0.262988,-0.743587,-0.146896,-0.067323,-0.100425,0.071572,-0.222507,...,0.117930,-0.324336,0.203228,0.444411,0.160399,-0.122193,-0.094062,-0.154989,0.303665,0.201930
3324,34435391,0.103390,0.656023,-0.262988,-0.743587,-0.146896,-0.067323,-0.100425,0.071572,-0.222507,...,0.117930,-0.324336,0.203228,0.444411,0.160399,-0.122193,-0.094062,-0.154989,0.303665,0.201930


In [8]:
# Tính toán độ tương đồng giữa các cặp POI

start_time = time.time()

# Chuẩn bị dữ liệu dạng dictionary để tìm kiếm O(1) và loại bỏ overhead của Pandas
categorical_dict = df_categorical_cols.set_index('poi_id').drop(columns=['name'], errors='ignore').to_dict(orient='index')
categorical_dict = {k: np.array(list(v.values()), dtype=bool) for k, v in categorical_dict.items()}

numerical_dict = df_numerical_cols.set_index('poi_id').drop(columns=['name'], errors='ignore').to_dict(orient='index')
numerical_dict = {k: np.array(list(v.values()), dtype=float) for k, v in numerical_dict.items()}

# Trích xuất và tiền chuẩn hóa vector L2 cho độ tương đồng Cosine nhanh
textual_dict = df_textual_cols.set_index('poi_id').to_dict(orient='index')
textual_dict_norm = {}
for k, v in textual_dict.items():
    arr = np.array(list(v.values()), dtype=float)
    norm = np.linalg.norm(arr)
    textual_dict_norm[k] = arr / norm if norm > 0 else arr

num_cat_cols = len(categorical_cols) # 3
num_num_cols = len(numerical_cols) # 8
num_text_cols = len(textual_cols) # 1
total_weights = num_cat_cols + num_num_cols + num_text_cols

poi_ids = df_distinct_pois['poi_id'].tolist()
n_pois = len(poi_ids)

print(f"Bắt đầu tính toán độ tương đồng cho {n_pois * (n_pois - 1) // 2} cặp...")

similarity_pairs = []

# Vòng lặp tối ưu hóa bằng NumPy
for i in range(n_pois):
    poi1_id = poi_ids[i]
    
    # Lấy sẵn dữ liệu của poi1
    p1_cat = categorical_dict[poi1_id]
    p1_num = numerical_dict[poi1_id]
    p1_text = textual_dict_norm[poi1_id]
    
    for j in range(i + 1, n_pois):
        poi2_id = poi_ids[j]
        
        p2_cat = categorical_dict[poi2_id]
        p2_num = numerical_dict[poi2_id]
        p2_text = textual_dict_norm[poi2_id]
        
        # 1. Jaccard Similarity cho thuộc tính phân loại
        intersection = np.sum(p1_cat & p2_cat)
        union = np.sum(p1_cat | p2_cat)
        cat_cols_similarity = intersection / union if union != 0 else 0.0
        
        # 2. Euclidean Similarity cho thuộc tính số
        euclidean_distance = np.linalg.norm(p1_num - p2_num)
        num_cols_similarity = 1 / (1 + euclidean_distance)
        
        # 3. Cosine Similarity cho thuộc tính văn bản (đã chuẩn hóa L2 nên chỉ cần nhân vô hướng)
        text_cols_similarity = np.dot(p1_text, p2_text)
        
        # 4. Trọng số tổng hợp
        similarity = (num_cat_cols * cat_cols_similarity + 
                      num_num_cols * num_cols_similarity + 
                      num_text_cols * text_cols_similarity) / total_weights
        
        # Chỉ lưu các cặp có độ tương đồng lớn hơn hoặc bằng 0.5
        if similarity >= 0.5:
            similarity_pairs.append((poi1_id, poi2_id, similarity))

# Tạo DataFrame kết quả
df_similarity = pd.DataFrame(similarity_pairs, columns=['poi1_id', 'poi2_id', 'Similarity'])

# Sắp xếp theo mức độ tương đồng giảm dần
df_similarity = df_similarity.sort_values(by='Similarity', ascending=False).reset_index(drop=True)

end_time = time.time()
print(f"Hoàn thành tính toán trong {end_time - start_time:.2f} giây!")
print(f"Số lượng cặp POI tương đồng (>= 0.5): {len(df_similarity)}")

df_similarity

Bắt đầu tính toán độ tương đồng cho 5529475 cặp...
Hoàn thành tính toán trong 62.48 giây!
Số lượng cặp POI tương đồng (>= 0.5): 3293643


,poi1_id,poi2_id,Similarity
0,25550706,34367010,1.000000
1,25550706,34311205,1.000000
2,20902795,27417855,1.000000
3,4795443,28288981,1.000000
4,33072381,34311205,1.000000
...,...,...,...
3293638,16657089,15334746,0.500001
3293639,15746305,21251594,0.500001
3293640,11488943,9715545,0.500001
3293641,3338621,21251594,0.500000


Chúng ta đã thiết lập điểm cắt độ tương đồng nhỏ hơn 0.5.

Chúng ta đã tìm thấy 3293643 cặp (quan hệ vô hướng) POI tương đồng.

# Ghi ngược độ tương đồng vào cơ sở dữ liệu

Tạo một quan hệ mới cho các truy vấn gợi ý trong tương lai, giúp truy xuất thông tin dễ dàng bằng câu lệnh cypher đơn giản.

In [ ]:
# Chạy cell này để xóa quan hệ cũ
# total_deleted = 0
# while True:
#     query = "MATCH ()-[r:CBF_SIMILAR]->() WITH r LIMIT 50000 DELETE r RETURN count(r) as deleted_count"
#     result = graph.run(query).data()
#     deleted = result[0]['deleted_count']
#     total_deleted += deleted
#     if deleted == 0:
#         break
# print(f"Đã dọn sạch {total_deleted} quan hệ cũ!")


Đã dọn sạch 6955730 quan hệ cũ!


In [9]:
# Chuyển DataFrame thành list of dicts và lưu mối quan hệ vào Neo4j
# Đã lưu quan hệ vào Neo4j thì không chạy lại cell này nữa !!!
data_list = df_similarity.to_dict(orient='records')

batch_query = """
UNWIND $rows AS row
MATCH (poi1:Poi {id: row.poi1_id})
MATCH (poi2:Poi {id: row.poi2_id})
MERGE (poi1)-[s1:CBF_SIMILAR]->(poi2)
ON CREATE SET s1.score = row.Similarity
MERGE (poi1)<-[s2:CBF_SIMILAR]-(poi2)
ON CREATE SET s2.score = row.Similarity
"""

batch_size = 50000
total_rows = len(data_list)

for i in range(0, total_rows, batch_size):
    batch = data_list[i : i + batch_size]
    graph.run(batch_query, rows=batch)
    print(f"Đã ghi xong từ {i} đến {min(i + batch_size, total_rows)}")

print("Hoàn thành ghi toàn bộ dữ liệu quan hệ vào Neo4j!")

Đã ghi xong từ 0 đến 50000
Đã ghi xong từ 50000 đến 100000
Đã ghi xong từ 100000 đến 150000
Đã ghi xong từ 150000 đến 200000
Đã ghi xong từ 200000 đến 250000
Đã ghi xong từ 250000 đến 300000
Đã ghi xong từ 300000 đến 350000
Đã ghi xong từ 350000 đến 400000
Đã ghi xong từ 400000 đến 450000
Đã ghi xong từ 450000 đến 500000
Đã ghi xong từ 500000 đến 550000
Đã ghi xong từ 550000 đến 600000
Đã ghi xong từ 600000 đến 650000
Đã ghi xong từ 650000 đến 700000
Đã ghi xong từ 700000 đến 750000
Đã ghi xong từ 750000 đến 800000
Đã ghi xong từ 800000 đến 850000
Đã ghi xong từ 850000 đến 900000
Đã ghi xong từ 900000 đến 950000
Đã ghi xong từ 950000 đến 1000000
Đã ghi xong từ 1000000 đến 1050000
Đã ghi xong từ 1050000 đến 1100000
Đã ghi xong từ 1100000 đến 1150000
Đã ghi xong từ 1150000 đến 1200000
Đã ghi xong từ 1200000 đến 1250000
Đã ghi xong từ 1250000 đến 1300000
Đã ghi xong từ 1300000 đến 1350000
Đã ghi xong từ 1350000 đến 1400000
Đã ghi xong từ 1400000 đến 1450000
Đã ghi xong từ 1450000 đến 1500

# Exploring the results

Kiểm tra kết quả bằng cách sử dụng Cypher. 

Sử dụng kiểu quan hệ `SIMILAR` để lọc ra các mối quan hệ.

In [10]:
# tìm độ tương đồng lớn nhất giữa các cặp
gds.run_cypher(
    """
        MATCH (p1:Poi)-[r:CBF_SIMILAR]->(p2:Poi)
        RETURN p1.name AS poi1, p2.name AS poi2, r.score AS similarity
        ORDER BY similarity DESC, poi1, poi2
        LIMIT 10
    """
)

,poi1,poi2,similarity
0,67 Percent Store,Ninh Khuong Hand Embroidery,1.0
1,67 Percent Store,Pearl Shop - Gemmy Store,1.0
2,67 Percent Store,Real Clothes,1.0
3,67 Percent Store,Trang Sức Thủ Công Ssc,1.0
4,6Saigon Leather,Aesir Studios,1.0
5,6Saigon Leather,Chus Authentic Souvenir,1.0
6,6Saigon Leather,Convenient Store de Tham street,1.0
7,6Saigon Leather,Dati,1.0
8,6Saigon Leather,Fzone Store,1.0
9,6Saigon Leather,Hamong,1.0


In [11]:
# tìm độ tương đồng thấp nhất giữa các cặp
gds.run_cypher(
    """
        MATCH (p1:Poi)-[r:CBF_SIMILAR]->(p2:Poi)
        RETURN p1.name AS poi1, p2.name AS poi2, r.score AS similarity
        ORDER BY similarity ASC, poi1, poi2
        LIMIT 10
    """
)

,poi1,poi2,similarity
0,Lush,Saigon DMC Travel,0.500000
1,Saigon DMC Travel,Lush,0.500000
2,Arte,Usagi Tailor,0.500000
3,Usagi Tailor,Arte,0.500000
4,Saigon Private Cars,Ảnh Việt Tourist,0.500001
5,Ảnh Việt Tourist,Saigon Private Cars,0.500001
6,Arte,Vietnam Tourist Sim,0.500001
7,Vietnam Tourist Sim,Arte,0.500001
8,Bat Trang Handcrafted Ceramics Sai Gon,Công Viên Đầm Sen,0.500001
9,Công Viên Đầm Sen,Bat Trang Handcrafted Ceramics Sai Gon,0.500001


## Tiến hành gợi ý

Tạo gợi ý POI cho POI mục tiêu dựa trên độ tương đồng bằng truy vấn Cypher đơn giản.

In [42]:
# HÀM: Tạo gợi ý dựa trên phương pháp Lọc dựa trên nội dung - Độ tương đồng nút (Node Similarity)
# INPUT: poi_id
# OUTPUT: dataframe[poi_id, rec_poi_id]

# Thực nghiệm lần 1:
# Precision Score: 0.03188893506911392
# Recall Score: 0.9575281803542673
# Coverage Score: 1.0
# F1 Score: 0.06172230800171271

# Thực nghiệm lần 2 (k = 5):
# Precision Score: 0.496551724137931
# Recall Score: 0.028985507246376812
# Coverage Score: 0.06254738438210766
# F1 Score: 0.05477367820464055

# Thực nghiệm lần 3 (k = 10):
# Precision Score: 0.47758620689655173
# Recall Score: 0.05575684380032206
# Coverage Score: 0.10045489006823351
# F1 Score: 0.09985580389329488

# Áp dung phobert để embedding thay vì BoW
# Thực nghiệm lần 1:
# Precision Score: 0.03248783423159437
# Recall Score: 0.9997987117552335
# Coverage Score: 1.0
# F1 Score: 0.06293077234948308

# Thực nghiệm lần 2 (k = 5):
# Precision Score: 0.5103448275862069
# Recall Score: 0.029790660225442835
# Coverage Score: 0.060272934040940106
# F1 Score: 0.05629516926588056

# Thực nghiệm lần 3 (k = 10):
# Precision Score: 0.4982758620689655
# Recall Score: 0.05817230273752013
# Coverage Score: 0.09969673995451099
# F1 Score: 0.10418168709444846

# Crawl lại bộ dữ liệu lần 2 (Bổ sung field description)
# Thực nghiệm lần 1:
# Precision Score: 0.028342405325465288
# Recall Score: 0.8062848873268554
# Coverage Score: 0.9990980156343957
# F1 Score: 0.05475989890480202


# Thực nghiệm lần 2 (k = 5):
# Precision Score: 0.40484429065743943
# Recall Score: 0.024188546619805663
# Coverage Score: 0.06043295249549008
# F1 Score: 0.045649629340616456


# Thực nghiệm lần 3 (k = 10):
# Precision Score: 0.3462214411247803
# Recall Score: 0.04072772379574116
# Coverage Score: 0.10102224894768491
# F1 Score: 0.072881982981872

def similar_poi_recommendation(poi_id, k=10):
    result = gds.run_cypher(
        """
            MATCH (p1:Poi {id: $target_poi})-[s:CBF_SIMILAR]->(p2:Poi)
            RETURN p1.id as poi_id, p2.id as rec_poi_id
            ORDER BY s.score DESC
        """, params = {'target_poi': poi_id}
    )
    result = result.drop_duplicates()
    return result.head(k)

In [43]:
# ID của POI mục tiêu là "Landmark 81 SkyView"
poi_id = 18314088

similar_poi_recommendation(poi_id)

,poi_id,rec_poi_id
0,18314088,16710665
1,18314088,15349040
2,18314088,26467466
3,18314088,6131915
4,18314088,26916395
5,18314088,6890386
6,18314088,19983914
7,18314088,26325096
8,18314088,18178406
9,18314088,33000429


# Đánh giá

In [44]:
# DataFrame các POI
df_pois = gds.run_cypher("""\
    MATCH (poi:Poi)    
    RETURN poi.id
    """)

df_pois

,poi.id
0,311103
1,2005826
2,311087
3,4542125
4,311089
...,...
3321,34509929
3322,34508995
3323,34515006
3324,34435391


In [45]:
# DataFrame các đánh giá

df_reviews = gds.run_cypher("""\
    MATCH (user:User)-[review:REVIEWED]->(poi:Poi)
    RETURN user.id AS user_id, poi.id AS poi_id
    """)

df_reviews

,user_id,poi_id
0,1,311103
1,2,311103
2,3,311103
3,4,311103
4,5,311103
...,...,...
23943,21311,15129760
23944,21312,15129760
23945,21313,15129760
23946,21314,15129760


In [46]:
# Nhóm theo 'user_id' và đếm số lần xuất hiện
user_counts = df_reviews.groupby('user_id').size()

# Lọc ra các người dùng có ít hơn 5 lần xuất hiện
valid_users = user_counts[user_counts >= 5].index

# Lọc DataFrame gốc dựa trên danh sách người dùng hợp lệ
filtered_df_reviews = df_reviews[df_reviews['user_id'].isin(valid_users)].copy()
filtered_df_reviews

,user_id,poi_id
50,51,311103
51,52,311103
103,104,311103
116,117,311103
118,119,311103
...,...,...
23570,481,12829864
23571,1303,12829864
23573,2886,12829864
23613,1246,9974597


In [47]:
# Chia tập dữ liệu thành 90% tập huấn luyện (training) và 10% tập kiểm tra (test)
df_train, df_test = train_test_split(filtered_df_reviews, test_size=0.1, random_state=100)

df_train

,user_id,poi_id
23463,181,9806390
2694,473,311094
3279,1177,1910195
2331,1334,10836601
8110,51,16657089
...,...,...
4403,284,311092
555,493,4542125
2215,1459,4552853
745,52,2037764


In [48]:
df_test

,user_id,poi_id
3284,1253,1910195
5049,683,552637
14873,606,10044301
1854,472,317896
2336,1415,10836601
...,...,...
1378,1006,10005057
23298,181,10388517
23338,474,1809055
545,484,311087


In [49]:
# Lấy gợi ý cho từng dòng trong tập kiểm tra

df_all_retrieved = pd.DataFrame()
for index, row in df_test.iterrows():
    poi_id = row['poi_id']

    recommended_interactions = similar_poi_recommendation(poi_id)

    # Nối các tương tác gợi ý với test_recommendations
    df_all_retrieved = pd.concat([df_all_retrieved, recommended_interactions], ignore_index=True)

# Loại bỏ cột trùng lặp
df_all_retrieved = df_all_retrieved.drop_duplicates()

df_all_retrieved

,poi_id,rec_poi_id
0,1910195,9596399
1,1910195,8616822
2,1910195,8071799
3,1910195,12878922
4,1910195,20051280
...,...,...
903,1809055,25307438
904,1809055,33761555
905,1809055,32979951
906,1809055,23997911


In [50]:
# Trích xuất các tương tác thực tế của tất cả các cặp POI được đánh giá bởi cùng người dùng

# Nhóm theo user_id và tổng hợp các poi_id thành một danh sách
grouped = df_reviews.groupby('user_id')['poi_id'].apply(list)

# Khởi tạo một DataFrame trống cho kết quả
df_true_interactions = pd.DataFrame(columns=['poi_id', 'rec_poi_id'])

# Lặp qua từng nhóm
for user_id, poi_ids in grouped.items():
    # Tạo các cặp poi_id và rec_poi_id cho từng người dùng
    pairs = [(poi_id, other_poi_id) for poi_id in poi_ids for other_poi_id in poi_ids if poi_id != other_poi_id]
    df_pairs = pd.DataFrame(pairs, columns=['poi_id', 'rec_poi_id'])
    # Thêm các cặp vào DataFrame kết quả
    df_true_interactions = pd.concat([df_true_interactions, df_pairs], ignore_index=True)

df_true_interactions = df_true_interactions.drop_duplicates()
# Hiển thị DataFrame kết quả
df_true_interactions

,poi_id,rec_poi_id
0,311103,2005826
1,311103,311087
2,311103,8587831
3,2005826,311103
4,2005826,311087
...,...,...
19089,11435998,10813505
19090,2493165,17376382
19091,17376382,2493165
19092,13613302,10836471


In [51]:
# Lấy tất cả các thực thể liên quan bằng cách gộp các tương tác thực tế và thực thể kiểm tra
df_all_relevant = pd.merge(df_true_interactions, df_test, on=['poi_id'], how='inner')
# Loại bỏ cột user_id
df_all_relevant = df_all_relevant.drop(columns=['user_id'])
# kiểm tra cặp POI không phải là cùng một POI
df_all_relevant = df_all_relevant[df_all_relevant['poi_id'] != df_all_relevant['rec_poi_id']]
# Loại bỏ trùng lặp
df_all_relevant = df_all_relevant.drop_duplicates()

df_all_relevant

,poi_id,rec_poi_id
0,2005826,311103
2,2005826,311087
4,2005826,8587831
6,311087,311103
10,311087,2005826
...,...,...
10116,10044301,13873247
10117,13447187,34239306
10118,13447187,13961526
10119,1949590,16698454


In [52]:
# Lấy tất cả các thực thể gợi ý liên quan bằng cách gộp tương tác thực tế và tương tác gợi ý
df_retrived_relevant = pd.merge(df_all_retrieved, df_all_relevant, on=['poi_id', 'rec_poi_id'], how='inner')

df_retrived_relevant

,poi_id,rec_poi_id
0,552637,7714795
1,552637,9806515
2,552637,8856733
3,552637,10085046
4,552637,311105
...,...,...
192,1784754,4597843
193,1784754,311100
194,2414430,14928513
195,10388517,9806390


In [53]:
# Tính chỉ số độ chính xác (precision score)
relevant_retrieved = df_retrived_relevant.shape[0]
all_retrived = df_all_retrieved.shape[0]

precision = relevant_retrieved / all_retrived

print(f'Precision Score: {precision}')

Precision Score: 0.3462214411247803


In [54]:
# Tính chỉ số độ bao phủ (recall score)
relevant_retrieved = df_retrived_relevant.shape[0]
all_relevant = df_all_relevant.shape[0]

recall = relevant_retrieved / all_relevant
print(f'Recall Score: {recall}')

Recall Score: 0.04072772379574116


In [55]:
# Tính chỉ số độ phủ (coverage score)
num_recommended_pois = df_all_retrieved['rec_poi_id'].nunique()
num_all_pois = df_pois.shape[0]

coverage = num_recommended_pois / num_all_pois
print(f'Coverage Score: {coverage}')

Coverage Score: 0.10102224894768491


In [56]:
# Tính điểm F1 (F1 score)
f1 = (2 * precision * recall) / (precision + recall)
print(f'F1 Score: {f1}')

F1 Score: 0.072881982981872
